# Week 2 — Find the craters
*Albania Now — a Free Focus program, built with Chicago First. Run cells top to bottom. First: **File → Save a copy in Drive**.*

## 1. The terrain (same generator as last week)

In [ ]:
import numpy as np

def make_terrain(craters, size=90, seed=1, noise=15, specks=0.006):
    """A synthetic orbital image: bright plain, shadowed crater bowls,
    sunlit rims, camera noise. craters = list of (cx, cy, r)."""
    rng = np.random.default_rng(seed)
    img = 180 + rng.integers(-noise, noise, (size, size)).astype(float)
    yy, xx = np.mgrid[0:size, 0:size]
    for cx, cy, r in craters:
        d = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
        bowl = d < r
        img[bowl & (xx < cx)] = 55 + rng.integers(0, 20, int((bowl & (xx < cx)).sum()))
        img[bowl & (xx >= cx)] = 225
        img[(d >= r) & (d < r + 1.5) & (xx > cx)] = 240
    sp = rng.random((size, size)) < specks
    img[sp] = 45
    return np.clip(img, 0, 255)

CRATERS = [(20, 18, 8), (55, 25, 11), (75, 70, 7), (30, 60, 9), (62, 48, 5)]
img = make_terrain(CRATERS)
print(img.shape, "pixels, values", int(img.min()), "to", int(img.max()))

## 2. Threshold — one comparison, every pixel
The histogram's valley sat near 100. Everything below it becomes True.

In [ ]:
import matplotlib.pyplot as plt

THRESHOLD = 100
mask = img < THRESHOLD
print(mask.sum(), "shadow pixels of", mask.size)

plt.figure(figsize=(5, 5))
plt.imshow(mask, cmap="gray")
plt.title("True = candidate shadow")
plt.show()

## 3. Group touching pixels into blobs

In [ ]:
def find_blobs(mask):
    """Group touching True pixels into blobs (the paint-bucket trick)."""
    seen = np.zeros_like(mask, dtype=bool)
    blobs = []
    H, W = mask.shape
    for y0 in range(H):
        for x0 in range(W):
            if mask[y0, x0] and not seen[y0, x0]:
                stack, px = [(y0, x0)], []
                seen[y0, x0] = True
                while stack:
                    y, x = stack.pop()
                    px.append((y, x))
                    for dy, dx in ((1,0), (-1,0), (0,1), (0,-1)):
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < H and 0 <= nx < W and mask[ny, nx] and not seen[ny, nx]:
                            seen[ny, nx] = True
                            stack.append((ny, nx))
                blobs.append(px)
    return blobs

In [ ]:
blobs = find_blobs(mask)
print(len(blobs), "blobs before any filtering")
print("blob sizes:", sorted(len(b) for b in blobs)[::-1][:12], "...")

## 4. The size filter — specks are not craters

In [ ]:
MIN_SIZE = 12
craters = [b for b in blobs if len(b) >= MIN_SIZE]
print(len(blobs), "blobs →", len(craters), "detections after the size filter")
print("True crater count in this terrain: 5")

## 5. See the detections

In [ ]:
overlay = np.stack([img, img, img], axis=-1) / 255
for b in craters:
    for y, x in b:
        overlay[y, x] = [0.85, 0.16, 0.12]
plt.figure(figsize=(5, 5))
plt.imshow(overlay)
plt.title(f"{len(craters)} detections")
plt.show()

## 6. The build — a field you haven't seen
A fresh terrain, crater count hidden. Tune THRESHOLD and MIN_SIZE, report the count, and defend both settings in two sentences (histogram valley; what the size filter costs).

**Turn-in:** detection image + settings + defense.

In [ ]:
secret = make_terrain([(15, 15, 7), (70, 20, 9), (45, 45, 12),
                       (20, 75, 6), (78, 78, 8), (60, 68, 4)],
                      seed=42, noise=18, specks=0.01)
# your detector here — histogram first, then threshold, blobs, filter, count
